In [9]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

In [6]:


# --- Step 1: Set path to your downloaded file ---
home = os.path.expanduser('~')
file_path = os.path.join(home, 'Downloads', 'Loughran-McDonald_10X_DocumentDictionaries_1993-2024.txt')

# --- Constants ---
apple_cik = '320193'  # No leading zeros
target_form = '10-K'
vocab_size = 100000  # Adjust based on your dictionary size

# --- Functions ---
def parse_word_counts(wordcount_part):
    counts = {}
    for pair in wordcount_part.strip().split(','):
        if ':' in pair:
            try:
                seq, count = map(int, pair.split(':'))
                counts[seq] = count
            except ValueError:
                continue  # Skip bad data
    return counts

def filing_to_vector(word_counts, vocab_size):
    vector = np.zeros(vocab_size)
    for seq, count in word_counts.items():
        if seq < vocab_size:
            vector[seq] = count
    return vector

# --- Step 2: Read file and collect Apple 10-K filings ---
filings = []

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            header_part, wordcount_part = line.strip().split('|', 1)
            fields = header_part.split(',')
            cik = fields[0].lstrip('0')  # Remove leading zeros
            form_type = fields[4]
            if cik == apple_cik and form_type == target_form:
                period_end = int(fields[3])
                word_counts = parse_word_counts(wordcount_part)
                filings.append((period_end, word_counts))
        except Exception as e:
            continue  # Skip malformed lines

# --- Step 3: Sort by year ---
filings.sort(key=lambda x: x[0])  # sort by period_end

# --- Step 4: Compare each year with the previous and store results ---
previous_vector = None
previous_year = None
results = []  # list to collect (CIK, Year, Distance)

for period_end, word_counts in filings:
    year = int(str(period_end)[:4])
    current_vector = filing_to_vector(word_counts, vocab_size)

    if previous_vector is not None:
        cos_sim = cosine_similarity([previous_vector], [current_vector])[0][0]
        cos_distance = 1 - cos_sim
        results.append((apple_cik, year, cos_distance))

    previous_vector = current_vector
    previous_year = year

# --- Step 5: Filter only past 20 years ---
current_year = 2024  # You can adjust if needed
earliest_year = current_year - 33  # So keep 1993 and newer

filtered_results = [(cik, year, distance) for (cik, year, distance) in results if year >= earliest_year]

# --- Step 6: Print the table ---
print("CIK,Year,Cosine Distance")

for row in filtered_results:
    cik, year, distance = row
    print(f"{cik},{year},{distance:.4f}")



CIK,Year,Cosine Distance
320193,1995,0.0035
320193,1996,0.0155
320193,1997,0.0160
320193,1999,0.0087
320193,2000,0.0338
320193,2002,0.0052
320193,2003,0.0121
320193,2004,0.0144
320193,2005,0.0085
320193,2006,0.0074
320193,2007,0.0122
320193,2008,0.0163
320193,2009,0.0013
320193,2010,0.0046
320193,2011,0.0009
320193,2012,0.0037
320193,2013,0.0011
320193,2014,0.0068
320193,2015,0.0063
320193,2016,0.0074
320193,2017,0.0040
320193,2018,0.0048
320193,2019,0.0079
320193,2020,0.0004
320193,2021,0.0057
320193,2022,0.0003
320193,2023,0.0003
320193,2024,0.0151


In [18]:
filtered_results
filtered_df = pd.DataFrame(filtered_results)

filtered_df = filtered_df.rename(columns={0: 'CIK', 1: 'Year', 2: 'Cosine Value'})
print(filtered_df.head())


      CIK  Year  Cosine Value
0  320193  1995      0.003520
1  320193  1996      0.015456
2  320193  1997      0.015980
3  320193  1999      0.008736
4  320193  2000      0.033792


In [14]:
import csv
sp500 = pd.read_csv('inputs/sp500.csv')
sp500

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [32]:
filtered_df['CIK'] = filtered_df['CIK'].astype(str)
sp500['CIK'] = sp500['CIK'].astype(str)
merged_df = pd.merge(filtered_df, sp500[['CIK', 'Symbol']], on='CIK', how='left')
filtered_df['Symbol'] = merged_df['Symbol']
print(filtered_df.head())


KeyError: 'Symbol'

In [15]:

crsp_monthly = pd.read_csv('inputs/crsp_data.csv')
crsp_monthly

,permno,date,ret
0,10000,1986-01-31,NaN
1,10000,1986-02-28,-25.7143
2,10000,1986-03-31,36.5385
3,10000,1986-04-30,-9.8592
4,10000,1986-05-30,-22.2656
...,...,...,...
4047625,93436,2024-08-30,-7.7391
4047626,93436,2024-09-30,22.1942
4047627,93436,2024-10-31,-4.5025
4047628,93436,2024-11-29,38.1469


In [26]:
crsp_monthly['date'] = pd.to_datetime(crsp_monthly['date'])
crsp_monthly['filing_month'] = crsp_monthly['date'].dt.month
crsp_monthly

,permno,date,ret,filing_month
0,10000,1986-01-31,NaN,1
1,10000,1986-02-28,-25.7143,2
2,10000,1986-03-31,36.5385,3
3,10000,1986-04-30,-9.8592,4
4,10000,1986-05-30,-22.2656,5
...,...,...,...,...
4047625,93436,2024-08-30,-7.7391,8
4047626,93436,2024-09-30,22.1942,9
4047627,93436,2024-10-31,-4.5025,10
4047628,93436,2024-11-29,38.1469,11


In [34]:
import os
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# --- Step 1: Set path to your downloaded file ---
home = os.path.expanduser('~')
file_path = os.path.join(home, 'Downloads', 'Loughran-McDonald_10X_DocumentDictionaries_1993-2024.txt')

# --- Constants ---
apple_cik = '320193'  # No leading zeros
target_form = '10-K'
vocab_size = 100000  # Adjust based on your dictionary size

# --- Functions ---
def parse_word_counts(wordcount_part):
    counts = {}
    for pair in wordcount_part.strip().split(','):
        if ':' in pair:
            try:
                seq, count = map(int, pair.split(':'))
                counts[seq] = count
            except ValueError:
                continue  # Skip bad data
    return counts

def filing_to_vector(word_counts, vocab_size):
    vector = np.zeros(vocab_size)
    for seq, count in word_counts.items():
        if seq < vocab_size:
            vector[seq] = count
    return vector

# --- Step 2: Read file and collect Apple 10-K filings with filing date ---
filings_data = []

with open(file_path, 'r', encoding='utf-8') as file:
    for line in file:
        try:
            header_part, wordcount_part = line.strip().split('|', 1)
            fields = header_part.split(',')
            cik = fields[0].lstrip('0')  # Remove leading zeros
            filing_date_str = fields[1]
            form_type = fields[4]
            if cik == apple_cik and form_type == target_form:
                period_end = int(fields[3])
                word_counts = parse_word_counts(wordcount_part)
                filings_data.append((cik, filing_date_str, period_end, word_counts))
        except Exception as e:
            continue  # Skip malformed lines

# --- Step 3: Sort by period end (year) ---
filings_data.sort(key=lambda x: x[2])  # sort by period_end

# --- Step 4: Compare each year with the previous and store results with filing year ---
previous_vector = None
previous_year = None
results = []  # list to collect (CIK, Year, FilingYear, Distance)

for cik, filing_date_str, period_end, word_counts in filings_data:
    year = int(str(period_end)[:4])
    filing_year = int(filing_date_str[:4])
    current_vector = filing_to_vector(word_counts, vocab_size)

    if previous_vector is not None:
        cos_sim = cosine_similarity([previous_vector], [current_vector])[0][0]
        cos_distance = 1 - cos_sim
        results.append((cik, year, filing_year, cos_distance))

    previous_vector = current_vector
    previous_year = year

# --- Step 5: Filter only past 33 years (1993 onwards for 2024) ---
current_year = 2024  # You can adjust if needed
earliest_year = current_year - 31  # Keep 1993 and newer (comparison starts from the second year)

filtered_results = [(cik, year, filing_year, distance) for (cik, year, filing_year, distance) in results if year >= earliest_year]

# --- Step 6: Create Pandas DataFrame ---
filtered_df = pd.DataFrame(filtered_results, columns=['CIK', 'Year', 'FilingYear', 'Cosine Distance'])

# --- Step 7: Print the DataFrame ---
print(filtered_df)


       CIK  Year  FilingYear  Cosine Distance
0   320193  1995        1995         0.003520
1   320193  1996        1996         0.015456
2   320193  1997        1997         0.015980
3   320193  1999        1999         0.008736
4   320193  2000        2000         0.033792
5   320193  2002        2002         0.005187
6   320193  2003        2003         0.012091
7   320193  2004        2004         0.014377
8   320193  2005        2005         0.008515
9   320193  2006        2006         0.007419
10  320193  2007        2007         0.012225
11  320193  2008        2008         0.016275
12  320193  2009        2009         0.001298
13  320193  2010        2010         0.004562
14  320193  2011        2011         0.000900
15  320193  2012        2012         0.003719
16  320193  2013        2013         0.001065
17  320193  2014        2014         0.006754
18  320193  2015        2015         0.006347
19  320193  2016        2016         0.007373
20  320193  2017        2017      